# 1. DLinear reconstruction on Weather

This notebook reconstructs the method and evaluation protocol from
*Are Transformers Effective for Time Series Forecasting?*

By the end we will have checked the dataset, chronological split,
train-only scaling, daily seasonal-naive baseline, DLinear architecture,
training objective, paper metrics, and our reproduced test result.

## Section 1 — Imports and reproducible configuration

The reusable implementation lives under `src/ts_project/`. Keeping the
data pipeline, model, and training loop outside the notebook prevents
accidental differences between the reconstruction and improvement.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ts_project.data import prepare_weather, build_weather_window_datasets
from ts_project.models import DLinear
from ts_project.training import seed_everything, train_forecaster

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "weather.csv"
INPUT_LENGTH = 336
HORIZON = 96
BATCH_SIZE = 16
SEED = 2021
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

## Section 2 — Load and audit Weather

Weather contains 21 meteorological variables sampled every ten minutes.
The supplied benchmark has one duplicate timestamp and one gap; we keep
its row order unchanged to match the paper's benchmark representation.

In [2]:
weather = prepare_weather(DATA_PATH)
split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "start": weather.split(name).index.min(),
            "end": weather.split(name).index.max(),
            "rows": len(weather.split(name)),
        }
        for name in ("train", "validation", "test")
    ]
)
print(f"Rows: {len(weather.raw):,}")
print(f"Variables: {len(weather.channel_names)}")
display(split_summary)

Rows: 52,696
Variables: 21


,split,start,end,rows
0,train,2020-01-01 00:10:00,2020-09-13 05:10:00,36887
1,validation,2020-09-13 05:20:00,2020-10-19 19:30:00,5270
2,test,2020-10-19 19:40:00,2021-01-01 00:00:00,10539


## Section 3 — Leakage-safe forecasting windows

Each example uses 336 past observations (56 hours) to predict all 96
future observations (16 hours) directly. Validation and test may use
earlier values as historical context, but every target stays inside its
own chronological partition. The scaler was fitted only on training rows.

In [3]:
datasets = build_weather_window_datasets(
    weather,
    input_length=INPUT_LENGTH,
    prediction_length=HORIZON,
)
loaders = {
    "train": DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
    "validation": DataLoader(datasets["validation"], batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
    "test": DataLoader(datasets["test"], batch_size=BATCH_SIZE, shuffle=False, drop_last=False),
}
pd.Series({name: len(dataset) for name, dataset in datasets.items()}, name="windows")

train         36456
validation     5175
test          10444
Name: windows, dtype: int64

## Section 4 — Original DLinear

DLinear estimates a trend with a centered 25-sample moving average and
defines the remainder as input minus trend. Two shared linear layers map
the complete 336-step trend and remainder directly to the 96-step future;
the forecasts are added. Training minimizes MSE with Adam.

In [4]:
seed_everything(SEED)
model = DLinear(
    input_length=INPUT_LENGTH,
    prediction_length=HORIZON,
    channels=len(weather.channel_names),
    moving_average=25,
    individual=False,
)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Trainable parameters: {parameter_count:,}")

Trainable parameters: 64,704


## Section 5 — Train and select by validation MSE

This cell takes roughly one to two minutes on the project GPU. Early
stopping restores the checkpoint with the lowest validation MSE. The test
set is not available to the training function.

In [5]:
seed_everything(SEED)
training = train_forecaster(
    model,
    loaders["train"],
    loaders["validation"],
    device=DEVICE,
    learning_rate=1e-4,
    learning_rate_schedule="type1",
    max_epochs=10,
    patience=3,
    verbose=True,
)
pd.DataFrame(training.history)

Epoch   1/10 | train MSE 0.506181 | validation MSE 0.433110 | lr 1.00e-04


Epoch   2/10 | train MSE 0.464539 | validation MSE 0.430417 | lr 1.00e-04


Epoch   3/10 | train MSE 0.457825 | validation MSE 0.426956 | lr 5.00e-05


Epoch   4/10 | train MSE 0.455450 | validation MSE 0.426791 | lr 2.50e-05


Epoch   5/10 | train MSE 0.454530 | validation MSE 0.427713 | lr 1.25e-05


Epoch   6/10 | train MSE 0.454033 | validation MSE 0.425668 | lr 6.25e-06


Epoch   7/10 | train MSE 0.453783 | validation MSE 0.426106 | lr 3.13e-06


Epoch   8/10 | train MSE 0.453680 | validation MSE 0.425862 | lr 1.56e-06


Epoch   9/10 | train MSE 0.452719 | validation MSE 0.426247 | lr 7.81e-07


,epoch,learning_rate,train_mse,validation_mse
0,1,1.000000e-04,0.506181,0.433110
1,2,1.000000e-04,0.464539,0.430417
2,3,5.000000e-05,0.457825,0.426956
3,4,2.500000e-05,0.455450,0.426791
4,5,1.250000e-05,0.454530,0.427713
5,6,6.250000e-06,0.454033,0.425668
6,7,3.125000e-06,0.453783,0.426106
7,8,1.562500e-06,0.453680,0.425862
8,9,7.812500e-07,0.452719,0.426247


## Section 6 — Evaluate MSE and MAE

MSE penalizes large errors quadratically; MAE reports the average absolute
error. Both are computed over every test window, future step, and variable
in standardized space, matching the paper.

In [6]:
@torch.inference_mode()
def evaluate(model, loader):
    squared_sum = absolute_sum = 0.0
    elements = 0
    model.eval()
    for inputs, targets in loader:
        errors = model(inputs.to(DEVICE)) - targets.to(DEVICE)
        squared_sum += errors.square().sum().item()
        absolute_sum += errors.abs().sum().item()
        elements += errors.numel()
    return {"MSE": squared_sum / elements, "MAE": absolute_sum / elements}

measured = evaluate(model, loaders["test"])
comparison = pd.DataFrame(
    [
        {"result": "Paper DLinear", "MSE": 0.176, "MAE": 0.237},
        {"result": "Our reconstruction", **measured},
    ]
)
comparison

,result,MSE,MAE
0,Paper DLinear,0.176000,0.237000
1,Our reconstruction,0.174205,0.233328


## Section 7 — Reconstruction conclusion

The expected seed-2021 result is approximately MSE 0.17421 and MAE
0.23333, very close to the paper's 0.176 and 0.237. This establishes a
faithful baseline before changing DLinear's decomposition.